In [49]:
import numpy as np
import pandas as pd

import circuit 
from bus import BusType



# Jacobian Tests

In [50]:
def _jacobian_labels(circuit):
    """Return (row_labels, col_labels) matching calc_jacobian()'s layout.

    Layout
    ------
    Rows : dP/<bus>  for every non-slack bus  (pvbuses order)
           dQ/<bus>  for every PQ bus          (pqbuses order)
    Cols : delta_<bus>   for every non-slack bus  (pvbuses order)
           voltmag_<bus> for every PQ bus          (pqbuses order)
    """
    buses, busnames, _, _, _ = circuit._get_data()
    pv_list = circuit._pv_buses(busnames, buses)
    pq_list = circuit._pq_buses(busnames, buses)

    row_labels = [f"dP/{n}" for n in pv_list] + [f"dQ/{n}" for n in pq_list]
    col_labels = [f"delta_{n}" for n in pv_list] + [f"voltmag_{n}" for n in pq_list]
    return row_labels, col_labels

In [51]:
def compare_jacobians(circuit, excel_path: str, tol: float = 1e-2):
    """
    Compare computed Jacobian against a reference Excel.
    Both matrices are aligned positionally — no label matching.

    Canonical order
    ---------------
    Rows : dP for non-slack buses (ascending bus number)
           dQ for PQ buses        (ascending bus number)
    Cols : |V| for PQ buses       (ascending bus number)
           delta for non-slack buses (ascending bus number)
    """
    buses, bus_names, ybus, angles, voltages = circuit._get_data()
    idx     = circuit._bus_index          # name -> 0-based insertion index
    pv_list = circuit._pv_buses(bus_names, buses)   # non-slack, already sorted
    pq_list = circuit._pq_buses(bus_names, buses)   # PQ only,   already sorted
    n_pv, n_pq = len(pv_list), len(pq_list)

    # 1-indexed bus numbers aligned with the reference Excel numbering
    bus_num   = {n: idx[n] + 1 for n in bus_names}
    slack_set = {bus_num[n] for n in bus_names
                 if buses[n].bus_type == BusType.Slack}
    pq_nums   = sorted(bus_num[n] for n in pq_list)
    pv_nums   = sorted(bus_num[n] for n in pv_list)

    # ── Generated Jacobian ──────────────────────────────────────────────
    # _calc_jacobian() col layout: [delta pv_list... | voltmag pq_list...]
    # Desired layout:              [voltmag pq_list... | delta pv_list...]
    J_raw  = circuit._calc_jacobian()
    col_perm = list(range(n_pv, n_pv + n_pq)) + list(range(n_pv))
    J_computed = J_raw[:, col_perm]

    # ── Reference Excel ─────────────────────────────────────────────────
    # header=1  → skip row-0 ("Bus","Unnamed:1",...) and use row-1 as header
    #             giving cols: "Name","Jacobian Equation","Angle Bus X","Volt Mag Bus X"
    # index_col=0 → bus numbers (1-5) become the index
    ref_raw = pd.read_excel(excel_path, header=1, index_col=0)

    # dP rows: "Real Power ..." entries, excluding the slack bus
    dp_mask = (ref_raw["Jacobian Equation"].str.contains("Real Power", na=False) &
               ~ref_raw.index.isin(slack_set))
    dp_part = ref_raw[dp_mask].sort_index()   # ascending bus number

    # dQ rows: "Reactive Power" entries (PQ buses only)
    dq_mask = ref_raw["Jacobian Equation"].str.contains("Reactive Power", na=False)
    dq_part = ref_raw[dq_mask].sort_index()   # ascending bus number

    # Select & reorder columns to match canonical [|V| PQ | delta non-slack]
    volt_cols  = [f"Volt Mag Bus {b}" for b in pq_nums]
    angle_cols = [f"Angle Bus {b}"    for b in pv_nums]
    J_ref = (pd.concat([dp_part, dq_part])[volt_cols + angle_cols]
               .astype(float)
               .fillna(0.0)      # NaN in reference = no coupling = 0
               .values)

    # ── Element-wise comparison ─────────────────────────────────────────
    diff  = J_computed - J_ref
    match = bool(np.all(np.abs(diff) <= tol))

    # Human-readable labels (for display only — not used for alignment)
    row_labels = ([f"dP/Bus{bus_num[n]}" for n in pv_list] +
                  [f"dQ/Bus{bus_num[n]}" for n in pq_list])
    col_labels = ([f"|V|_Bus{b}" for b in pq_nums] +
                  [f"d_Bus{b}"   for b in pv_nums])

    computed_df = pd.DataFrame(J_computed, index=row_labels, columns=col_labels)
    ref_df_out  = pd.DataFrame(J_ref,      index=row_labels, columns=col_labels)
    diff_df     = pd.DataFrame(diff,        index=row_labels, columns=col_labels)

    print("Computed Jacobian:\n", computed_df.round(3), "\n")
    print("Reference Jacobian:\n", ref_df_out.round(3), "\n")
    print(f"Match (tol={tol}): {match}  |  Max |diff| = {np.max(np.abs(diff)):.4f}")

    return match, diff_df, J_computed, J_ref

## example

In [52]:
case69 = circuit.case6_9()
ybus = case69.calc_ybus()

In [54]:
ybus = pd.read_csv('case69_ybus.csv')

In [56]:
compare_jacobians(case69, 'jacobian0_case69.xlsx')

Computed Jacobian:
          |V|_Bus2  |V|_Bus4  |V|_Bus5  d_Bus2   d_Bus3   d_Bus4   d_Bus5
dP/Bus2     2.678    -0.893    -1.786  29.759    0.000   -9.920  -19.839
dP/Bus3     0.000    -7.831     0.000   0.000  104.413 -104.413    0.000
dP/Bus4    -0.893    11.549    -3.571  -9.920 -104.413  154.011  -39.679
dP/Bus5    -1.786    -3.571     9.086 -19.839    0.000  -39.679  109.238
dQ/Bus2    27.159    -9.920   -19.839  -2.678   -0.000    0.893    1.786
dQ/Bus4    -9.920   141.907   -39.679   0.893    7.831  -12.295    3.571
dQ/Bus5   -19.839   -39.679   107.918   1.786   -0.000    3.571   -9.086 

Reference Jacobian:
          |V|_Bus2  |V|_Bus4  |V|_Bus5  d_Bus2  d_Bus3  d_Bus4  d_Bus5
dP/Bus2      2.68     -0.89     -1.79   29.76    0.00   -9.92  -19.84
dP/Bus3      0.00     -7.83      0.00    0.00  104.41 -104.41    0.00
dP/Bus4     -0.89     11.55     -3.57   -9.92 -104.41  154.01  -39.68
dP/Bus5     -1.79     -3.57      9.09  -19.84    0.00  -39.68  109.24
dQ/Bus2     27.16     -

(True,
          |V|_Bus2  |V|_Bus4  |V|_Bus5    d_Bus2    d_Bus3    d_Bus4    d_Bus5
 dP/Bus2 -0.001694 -0.002769  0.004463 -0.001048  0.000000  0.000349  0.000698
 dP/Bus3  0.000000 -0.000951  0.000000  0.000000  0.002679 -0.002679  0.000000
 dP/Bus4 -0.002769 -0.001011 -0.001074  0.000349 -0.002679  0.000933  0.001397
 dP/Bus5  0.004463 -0.001074 -0.004364  0.000698  0.000000  0.001397 -0.001772
 dQ/Bus2 -0.001048  0.000349  0.000698  0.001694 -0.000000  0.002769 -0.004463
 dQ/Bus4  0.000349 -0.003132  0.001397  0.002769  0.000951 -0.004794  0.001074
 dQ/Bus5  0.000698  0.001397 -0.001772 -0.004463 -0.000000  0.001074  0.004364,
 array([[   2.67830572,   -0.89276857,   -1.78553715,   29.75895248,
            0.        ,   -9.91965083,  -19.83930166],
        [   0.        ,   -7.8309509 ,    0.        ,    0.        ,
          104.41267868, -104.41267868,    0.        ],
        [  -0.89276857,   11.54898893,   -3.5710743 ,   -9.91965083,
         -104.41267868,  154.01093282,  -39

# Mismatch Compare

In [70]:
def compare_mismatch(
    circuit,
    excel_path: str,
    tol: float = 1.0,
    sheet_name: str = "Sheet1"
):
    """
    Compare the mismatch vector produced by _calc_mismatch() against a
    reference Excel file (format: mismatch0_case69-2.xlsx).

    The circuit mismatch is computed in per-unit and scaled by ``sbase``
    to match the MW / Mvar values stored in the reference file.

    Excel layout expected
    ---------------------
    Row 0  : "Bus" (merged title — skipped)
    Row 1  : Number | Name | Area Name | Type |
              Mismatch MW | Mismatch Mvar | Mismatch MVA   ← used as header
    Row 2+ : one row per bus

    Mismatch vector layout from _calc_mismatch()
    --------------------------------------------
    [ ΔP  for all non-slack buses  (sorted by bus index)  ]
    [ ΔQ  for PQ buses only        (sorted by bus index)  ]

    Reconstruction per bus
    ----------------------
    Slack   → ΔP_MW = 0,    ΔQ_Mvar = 0   (excluded from NR mismatch)
    PV      → ΔP_MW = f[i], ΔQ_Mvar = 0   (voltage is controlled)
    PQ      → ΔP_MW = f[i], ΔQ_Mvar = f[j]

    Parameters
    ----------
    circuit    : Circuit instance (Y-bus must be built, buses initialised)
    excel_path : Path to the reference .xlsx / .xls file
    tol        : Absolute tolerance in MW or Mvar (default 1.0)
    sheet_name : Excel sheet to read (default "Sheet1")

    Returns
    -------
    match     : bool        — True when every |computed - reference| ≤ tol
    result_df : DataFrame   — per-bus table with computed, reference and
                              difference columns for both ΔP and ΔQ
    diff_df   : DataFrame   — only the difference columns (ΔP_diff, ΔQ_diff)
    """
    sbase = grid_settings.sbase  # MVA base (typically 100)

    # ── 1. Compute mismatch and retrieve bus ordering ──────────────────────
    mismatch_pu = circuit._calc_mismatch()          # shape: (n_pv + n_pq,)

    buses, bus_names, _, _, _ = circuit._get_data()
    pv_list = circuit._pv_buses(bus_names, buses)   # all non-slack (PV + PQ)
    pq_list = circuit._pq_buses(bus_names, buses)   # PQ only

    n_pv = len(pv_list)

    # Scale from per-unit to MW / Mvar
    delta_P_mw   = mismatch_pu[:n_pv] * sbase       # one entry per non-slack bus
    delta_Q_mvar = mismatch_pu[n_pv:] * sbase        # one entry per PQ bus

    # Build per-bus lookup dictionaries  {bus_name: value}
    computed_DeltaP = {name: delta_P_mw[i]   for i, name in enumerate(pv_list)}
    computed_DeltaQ = {name: delta_Q_mvar[i] for i, name in enumerate(pq_list)}

    # ── 2. Read the reference Excel ────────────────────────────────────────
    ref = pd.read_excel(
        excel_path,
        sheet_name=sheet_name,
        header=1,           # row-1 ("Number", "Name", …) is the column header
        index_col=None,
    )
    ref.columns = [
        "Number", "Name", "Area", "Type",
        "Ref_DelP_MW", "Ref_DelQ_Mvar", "Ref_DelS_MVA",
    ]
    ref = ref.dropna(subset=["Name"])  # remove any stray blank rows

    # ── 3. Align computed values onto the reference table row by row ───────
    computed_DeltaP_col, computed_DeltaQ_col = [], []

    for _, row in ref.iterrows():
        name = str(row["Name"]).strip()
        bus  = buses.get(name)

        if bus is None:                          # bus not found in circuit
            computed_DeltaP_col.append(float("nan"))
            computed_DeltaQ_col.append(float("nan"))
            continue

        if bus.bus_type == BusType.Slack:        # slack excluded from NR mismatch
            computed_DeltaP_col.append(0.0)
            computed_DeltaQ_col.append(0.0)
        elif bus.bus_type == BusType.PV:         # PV: ΔP only, voltage is fixed
            computed_DeltaP_col.append(computed_DeltaP.get(name, float("nan")))
            computed_DeltaQ_col.append(0.0)
        else:                                    # PQ: both ΔP and ΔQ
            computed_DeltaP_col.append(computed_DeltaP.get(name, float("nan")))
            computed_DeltaQ_col.append(computed_DeltaQ.get(name, float("nan")))

    ref = ref.copy()
    ref["Comp_DelP_MW"]   = computed_DeltaP_col
    ref["Comp_DelQ_Mvar"] = computed_DeltaQ_col
    ref["Diff_DelP_MW"]   = ref["Comp_DelP_MW"]   - ref["Ref_DelP_MW"]
    ref["Diff_DelQ_Mvar"] = ref["Comp_DelQ_Mvar"] - ref["Ref_DelQ_Mvar"]

    # ── 4. Evaluate tolerance ──────────────────────────────────────────────
    diffs = ref[["Diff_DelP_MW", "Diff_DelQ_Mvar"]].dropna()
    match = bool((diffs.abs() <= tol).all().all())

    # ── 5. Build output DataFrames ─────────────────────────────────────────
    display_cols = [
        "Number", "Name", "Type",
        "Ref_DelP_MW",   "Comp_DelP_MW",   "Diff_DelP_MW",
        "Ref_DelQ_Mvar", "Comp_DelQ_Mvar", "Diff_DelQ_Mvar",
    ]
    result_df = ref[display_cols].reset_index(drop=True)
    diff_df   = ref[["Name", "Diff_DelP_MW", "Diff_DelQ_Mvar"]].reset_index(drop=True)

    return match, result_df, diff_df

In [71]:
case69._calc_mismatch()

array([-8.        ,  4.00845245,  0.37290242, -0.        , -1.5       ,
        6.05203232,  0.66      ])

In [72]:
ref_mismatch = pd.read_excel("mismatch0_case69.xlsx")

In [73]:
ref_mismatch

,Bus,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6
0,Number,Name,Area Name,Type,Mismatch MW,Mismatch Mvar,Mismatch MVA
1,1,One,1,Slack,0,0,0
2,2,Two,1,PQ,-800,-150,813.94
3,3,Three,1,PV,400.85,0,400.85
4,4,Four,1,PQ,37.29,605.2,606.35
5,5,Five,1,PQ,0,66,66


In [75]:
result, df1, df2 = compare_mismatch(
    case69,
    "mismatch0_case69.xlsx")

In [76]:
df1

,Number,Name,Type,Ref_DelP_MW,Comp_DelP_MW,Diff_DelP_MW,Ref_DelQ_Mvar,Comp_DelQ_Mvar,Diff_DelQ_Mvar
0,1,One,Slack,0.00,0.000000,0.000000,0.0,0.000000,0.000000e+00
1,2,Two,PQ,-800.00,-800.000000,0.000000,-150.0,-150.000000,8.526513e-14
2,3,Three,PV,400.85,400.845245,-0.004755,0.0,0.000000,0.000000e+00
3,4,Four,PQ,37.29,37.290242,0.000242,605.2,605.203232,3.231821e-03
4,5,Five,PQ,0.00,-0.000000,-0.000000,66.0,66.000000,-3.410605e-13


In [77]:
df2

,Name,Diff_DelP_MW,Diff_DelQ_Mvar
0,One,0.000000,0.000000e+00
1,Two,0.000000,8.526513e-14
2,Three,-0.004755,0.000000e+00
3,Four,0.000242,3.231821e-03
4,Five,-0.000000,-3.410605e-13
